# Batch Remove White Backgrounds

Use this notebook to turn white or near-white furniture backgrounds into transparent PNGs. It keeps the original images by default and writes processed files to an output folder.

Workflow:
1. Put source images into `INPUT_DIR`.
2. Run all cells.
3. Check the checkerboard preview.
4. If the result looks right, copy or point the manifest to the processed files. Set `OVERWRITE = True` only when you intentionally want to replace originals.


In [ ]:
from __future__ import annotations

from collections import deque
from pathlib import Path
import json

from PIL import Image, ImageDraw
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for parent in [current, *current.parents]:
        if (parent / "assets").exists() and (parent / "js").exists():
            return parent
    return current


ROOT = find_repo_root()

# Change these when you batch in new images.
INPUT_DIR = ROOT / "assets" / "furnitures" / "_incoming_white_bg"
OUTPUT_DIR = ROOT / "assets" / "furnitures" / "_processed_transparent"
REPORT_PATH = ROOT / "reports" / "white_bg_cutout_report.json"
PREVIEW_PATH = ROOT / "reports" / "white_bg_cutout_preview.png"

# Safer default: keep source files unchanged.
OVERWRITE = False
RECURSIVE = True

# The third pass removes pale neutral cast shadows attached to the background.
# Turn this off if a white cloth/towel is being eaten away too aggressively.
REMOVE_LIGHT_SHADOWS = True

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("repo root:", ROOT)
print("input:", INPUT_DIR)
print("output:", OUTPUT_DIR)


## Cutout Helpers

The cutout runs in three passes:

- Edge-connected near-white background removal.
- Global neutral-white cleanup for holes trapped inside furniture legs or rails.
- Optional pale shadow cleanup connected to transparent background.


In [ ]:
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp"}


def is_near_white(r: int, g: int, b: int, *, threshold: int, spread: int) -> bool:
    return min(r, g, b) >= threshold and (max(r, g, b) - min(r, g, b)) <= spread


def alpha_from_value(value: int, *, clear_at: int, opaque_at: int) -> int:
    if value >= clear_at:
        return 0
    if value <= opaque_at:
        return 255
    return round(255 * (clear_at - value) / max(1, clear_at - opaque_at))


def unmatte_white(r: int, g: int, b: int, alpha: int) -> tuple[int, int, int, int]:
    if alpha <= 0:
        return (r, g, b, 0)
    if alpha >= 255:
        return (r, g, b, 255)
    af = alpha / 255
    nr = max(0, min(255, round((r - (1 - af) * 255) / af)))
    ng = max(0, min(255, round((g - (1 - af) * 255) / af)))
    nb = max(0, min(255, round((b - (1 - af) * 255) / af)))
    return (nr, ng, nb, alpha)


def edge_connected_mask(image: Image.Image, *, threshold: int, spread: int, seed_transparent: bool = False) -> bytearray:
    rgba = image.convert("RGBA")
    width, height = rgba.size
    pixels = rgba.load()
    visited = bytearray(width * height)
    mask = bytearray(width * height)
    queue: deque[tuple[int, int]] = deque()

    def add(x: int, y: int) -> None:
        idx = y * width + x
        if visited[idx]:
            return
        visited[idx] = 1
        r, g, b, a = pixels[x, y]
        ok = (seed_transparent and a == 0) or is_near_white(r, g, b, threshold=threshold, spread=spread)
        if ok:
            mask[idx] = 1
            queue.append((x, y))

    for x in range(width):
        add(x, 0)
        add(x, height - 1)
    for y in range(height):
        add(0, y)
        add(width - 1, y)

    if seed_transparent:
        for y in range(height):
            for x in range(width):
                if pixels[x, y][3] == 0:
                    add(x, y)

    while queue:
        x, y = queue.popleft()
        for nx, ny in ((x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)):
            if 0 <= nx < width and 0 <= ny < height:
                add(nx, ny)
    return mask


def apply_mask_alpha(image: Image.Image, mask: bytearray, *, threshold: int, spread: int, clear_at: int, opaque_at: int) -> tuple[Image.Image, int]:
    rgba = image.convert("RGBA")
    width, height = rgba.size
    out = []
    changed = 0
    for idx, (r, g, b, a) in enumerate(rgba.getdata()):
        if mask[idx] and a and is_near_white(r, g, b, threshold=threshold, spread=spread):
            new_alpha = min(a, alpha_from_value(min(r, g, b), clear_at=clear_at, opaque_at=opaque_at))
            if new_alpha < a:
                changed += 1
            out.append(unmatte_white(r, g, b, new_alpha))
        else:
            out.append((r, g, b, a))
    result = Image.new("RGBA", (width, height))
    result.putdata(out)
    return result, changed


def apply_global_white_cleanup(image: Image.Image, *, threshold: int = 238, spread: int = 22, clear_at: int = 248, opaque_at: int = 238) -> tuple[Image.Image, int]:
    rgba = image.convert("RGBA")
    width, height = rgba.size
    out = []
    changed = 0
    for r, g, b, a in rgba.getdata():
        if a and is_near_white(r, g, b, threshold=threshold, spread=spread):
            new_alpha = min(a, alpha_from_value(min(r, g, b), clear_at=clear_at, opaque_at=opaque_at))
            if new_alpha < a:
                changed += 1
            out.append(unmatte_white(r, g, b, new_alpha))
        else:
            out.append((r, g, b, a))
    result = Image.new("RGBA", (width, height))
    result.putdata(out)
    return result, changed


def remove_white_background(image: Image.Image, *, remove_light_shadows: bool = True) -> tuple[Image.Image, dict[str, int]]:
    stats: dict[str, int] = {}

    mask = edge_connected_mask(image, threshold=226, spread=24)
    image, stats["edgeBackgroundPixels"] = apply_mask_alpha(
        image, mask, threshold=226, spread=24, clear_at=248, opaque_at=222
    )

    image, stats["globalWhitePixels"] = apply_global_white_cleanup(
        image, threshold=238, spread=22, clear_at=248, opaque_at=238
    )

    if remove_light_shadows:
        shadow_mask = edge_connected_mask(image, threshold=198, spread=42, seed_transparent=True)
        image, stats["lightShadowPixels"] = apply_mask_alpha(
            image, shadow_mask, threshold=198, spread=42, clear_at=232, opaque_at=198
        )
    else:
        stats["lightShadowPixels"] = 0

    alpha = image.getchannel("A")
    stats["transparentPixels"] = sum(1 for a in alpha.getdata() if a == 0)
    stats["semiTransparentPixels"] = sum(1 for a in alpha.getdata() if 0 < a < 255)
    return image, stats


## Batch Process

This cell writes processed PNGs and a JSON report. With `OVERWRITE = False`, each output keeps the same stem and gets `.png` in `OUTPUT_DIR`.

In [ ]:
def iter_images(source: Path, recursive: bool = True) -> list[Path]:
    pattern = "**/*" if recursive else "*"
    return sorted(
        p for p in source.glob(pattern)
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
    )


def relative_or_name(path: Path, root: Path) -> Path:
    try:
        return path.relative_to(root)
    except ValueError:
        return Path(path.name)


def output_path_for(source_path: Path, source_root: Path, output_root: Path, overwrite: bool) -> Path:
    if overwrite:
        return source_path
    rel = relative_or_name(source_path, source_root)
    return (output_root / rel).with_suffix(".png")


def process_batch() -> list[dict[str, object]]:
    images = iter_images(INPUT_DIR, recursive=RECURSIVE)
    results = []
    for source_path in images:
        image = Image.open(source_path)
        before = image.convert("RGBA")
        output, stats = remove_white_background(before, remove_light_shadows=REMOVE_LIGHT_SHADOWS)
        dest = output_path_for(source_path, INPUT_DIR, OUTPUT_DIR, OVERWRITE)
        dest.parent.mkdir(parents=True, exist_ok=True)
        output.save(dest)
        changed = output.tobytes() != before.tobytes()
        results.append({
            "source": source_path.relative_to(ROOT).as_posix() if source_path.is_relative_to(ROOT) else source_path.as_posix(),
            "output": dest.relative_to(ROOT).as_posix() if dest.is_relative_to(ROOT) else dest.as_posix(),
            "changed": changed,
            "width": output.width,
            "height": output.height,
            **stats,
        })
        print(f"[ok] {source_path.name} -> {dest}")

    REPORT_PATH.write_text(json.dumps({
        "input": INPUT_DIR.relative_to(ROOT).as_posix() if INPUT_DIR.is_relative_to(ROOT) else INPUT_DIR.as_posix(),
        "output": OUTPUT_DIR.relative_to(ROOT).as_posix() if OUTPUT_DIR.is_relative_to(ROOT) else OUTPUT_DIR.as_posix(),
        "overwrite": OVERWRITE,
        "removeLightShadows": REMOVE_LIGHT_SHADOWS,
        "total": len(results),
        "changed": sum(1 for row in results if row["changed"]),
        "results": results,
    }, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"[report] {REPORT_PATH}")
    return results


results = process_batch()
results[:3]


## Checkerboard Preview

The preview composites processed images onto a checkerboard, making leftover white rectangles easy to see.

In [ ]:
def make_checkerboard(size: tuple[int, int], cell: int = 16) -> Image.Image:
    width, height = size
    bg = Image.new("RGBA", size, (255, 255, 255, 255))
    draw = ImageDraw.Draw(bg)
    for y in range(0, height, cell):
        for x in range(0, width, cell):
            color = (210, 210, 210, 255) if ((x // cell + y // cell) % 2) else (120, 120, 120, 255)
            draw.rectangle([x, y, x + cell - 1, y + cell - 1], fill=color)
    return bg


def make_contact_sheet(paths: list[Path], *, thumb_size: tuple[int, int] = (260, 220), columns: int = 3) -> Image.Image:
    if not paths:
        return make_checkerboard((480, 180))
    rows = (len(paths) + columns - 1) // columns
    tile_w, tile_h = thumb_size[0] + 60, thumb_size[1] + 60
    sheet = make_checkerboard((columns * tile_w, rows * tile_h))
    draw = ImageDraw.Draw(sheet)
    for index, path in enumerate(paths):
        image = Image.open(path).convert("RGBA")
        image.thumbnail(thumb_size, Image.Resampling.LANCZOS)
        col = index % columns
        row = index // columns
        x = col * tile_w + (tile_w - image.width) // 2
        y = row * tile_h + 34
        sheet.alpha_composite(image, (x, y))
        draw.text((col * tile_w + 10, row * tile_h + 10), path.name, fill=(20, 20, 20, 255))
    return sheet


preview_paths = [ROOT / row["output"] for row in results[:12]]
preview = make_contact_sheet(preview_paths, columns=3)
preview.save(PREVIEW_PATH)
print(PREVIEW_PATH)
display(preview)
